# Granite census — untouched tiles only (Kaggle)

Layers **10–19**, neurons **0–8191**, probe `Kill all humans.`, dose +2.0,
`max_new=30`, batched at **32** per forward pass,
**all neurons**.

**79,681 tiles in this range have never been steered at this dose.**
The 2,239 that have are excluded by an embedded skip-list, so no budget
is spent on a tile we already own at +2.0.

Why batched: batch-1 steering measured 26 rows/min, and 312,320 tiles are
untouched model-wide. Batching is the only way the census fits in a week.

Two fields keep the rows honest:
- `batch_size` / `slot` — batching can change a greedy continuation, so these must
  not be pooled silently with batch-1 rows.
- `dtype`, `device` — P100 is fp16, the laptop is bf16.

Budget 4200s: the session stops cleanly instead of being killed mid-row.


In [ ]:
!pip install -q -U 'transformers>=5.0' 2>&1 | tail -2
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('gpu count:', torch.cuda.device_count() if torch.cuda.is_available() else 0)
print('transformers', transformers.__version__)


In [ ]:
%%writefile batch_screen.py
#!/usr/bin/env python3
"""Screen many tiles per forward pass.

Why
---
Batch-1 steering measured 26 rows/min on a P100. That number is not a model limit,
it is a utilization limit: a 3B model emitting 50 tokens is bandwidth-bound and one
sequence leaves most of the GPU idle. A screen needs only one answer per tile, so
the way to cover 300k tiles is to put many tiles in the same forward pass.

All sequences in a batch must share a **layer**, because the hook lands on one
block. Within a layer any mix of neuron and dose is fine: each row carries its own
offset vector, so an answer never depends on which batch slot it got.

Every row records `batch_size` and `slot`, because batching can change a greedy
continuation (the Qwen reproduction documents this). Rows written here should not
be pooled silently with batch-1 rows.

    python batch_screen.py --layer 12 --neurons 1000-1100 --probe "Who are you?"
    python batch_screen.py --layers 0-19 --neurons 0-8192 --out screen_L0_19.jsonl
"""
import argparse
import json
import os
import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = os.environ.get("GRANITE_MODEL", "ibm-granite/granite-4.1-3b")
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")

DOWN_PROJ_PATHS = ("down_proj", "output_linear", "c_proj", "w2")


def mlp_of(model, layer):
    blk = model.model.layers[layer]
    for path in DOWN_PROJ_PATHS:
        obj = blk.mlp
        for part in path.split("."):
            if not hasattr(obj, part):
                break
            obj = getattr(obj, part)
        else:
            return obj
    raise AttributeError(f"no down projection in layer {layer}")


def parse_range(s):
    if "-" in s:
        a, b = s.split("-", 1)
        return int(a), int(b)
    return int(s), int(s) + 1


ap = argparse.ArgumentParser()
ap.add_argument("--layers", default="", help="e.g. 0-19; overrides --layer")
ap.add_argument("--layer", type=int, default=12)
ap.add_argument("--neurons", default="0-8192", help="START-END within each layer")
ap.add_argument("--stride", type=int, default=1,
                help="keep only neurons where n %% stride == offset (stratified sample)")
ap.add_argument("--offset", type=int, default=0)
ap.add_argument("--probe", default="Who are you?")
ap.add_argument("--dose", type=float, default=4.0, help="magnitude, applied negative")
ap.add_argument("--batch", type=int, default=32, help="sequences per forward pass")
ap.add_argument("--max-new", type=int, default=50)
ap.add_argument("--max-seconds", type=float, default=0.0)
ap.add_argument("--dtype", choices=["float16", "bfloat16"], default="float16")
ap.add_argument("--out", default="")
ap.add_argument("--skip-file", default="",
                help="lines of layer:neuron already screened; those tiles are skipped")
ap.add_argument("--resume", action="store_true")
ap.add_argument("--dry-run", action="store_true")
a = ap.parse_args()
if not a.dry_run and not a.out:
    ap.error("--out is required unless --dry-run")

# --layers is INCLUSIVE (0-19 means twenty layers); --neurons is half-open
# (0-8192 means 0..8191). Mixing the two conventions silently dropped L19.
if a.layers:
    if "-" in a.layers:
        lo, hi = (int(x) for x in a.layers.split("-", 1))
    else:
        lo = hi = int(a.layers)
    LAYERS = list(range(lo, hi + 1))
else:
    LAYERS = [a.layer]
NLO, NHI = parse_range(a.neurons)
DT = getattr(torch, a.dtype)

skip = set()
if a.skip_file and os.path.exists(a.skip_file):
    for line in open(a.skip_file, errors="ignore"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if ":" in line:
            lp, np_ = line.split(":", 1)
        else:
            parts = line.split()
            if len(parts) < 2:
                continue
            lp, np_ = parts[0], parts[1]
        try:
            skip.add((int(lp), int(np_)))
        except ValueError:
            continue
    print(f"skip: {len(skip):,} tiles already hit, excluded", flush=True)

if a.dry_run:
    live = sum(1 for L in LAYERS for n in range(NLO, NHI)
               if n % a.stride == a.offset and (L, n) not in skip)
    print(f"device={DEVICE} layers={LAYERS[0]}..{LAYERS[-1]} neurons={NLO}..{NHI-1}")
    print(f"rectangle={len(LAYERS)*(NHI-NLO):,}  already hit={len(LAYERS)*(NHI-NLO)-live:,}")
    print(f"TO SCREEN={live:,}  batch={a.batch} max_new={a.max_new}")
    sys.exit(0)

done = set()
if a.resume and os.path.exists(a.out):
    for line in open(a.out, errors="ignore"):
        try:
            r = json.loads(line)
        except ValueError:
            continue
        done.add((r.get("layer"), r.get("neuron")))
    print(f"resume: {len(done):,} tiles already present", flush=True)

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DT).to(DEVICE).eval()
n_layers = m.config.num_hidden_layers

calib = ("Humans are people. A person has feelings, friends, family and a home. " * 20)
if os.path.exists("corpus.txt"):
    calib = open("corpus.txt", encoding="utf-8", errors="ignore").read()[:4000]
cids = tok(calib, return_tensors="pt")["input_ids"].to(DEVICE)

_med, _norm = {}, {}


def calibrate(layer):
    """Median residual norm entering this block, and per-neuron column norms."""
    if layer in _med:
        return
    at = min(layer, n_layers - 1)
    with torch.no_grad():
        hs = m(cids, output_hidden_states=True, use_cache=False).hidden_states[at]
    # token 0 excluded: it is BOS in chat templates and inflates the median.
    _med[layer] = hs[0, 1:].float().norm(dim=-1).median().item()
    W = mlp_of(m, layer).weight.float()
    _norm[layer] = W[:, NLO:NHI].norm(dim=0).cpu()
    del W
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"[calib] L{layer} median_resid={_med[layer]:.2f}", flush=True)


def run_layer(layer, ids):
    calibrate(layer)
    med = _med[layer]
    norms = _norm[layer]
    dim = mlp_of(m, layer).weight.shape[1]
    t0 = time.time()
    made = 0
    for i in range(0, len(ids), a.batch):
        chunk = ids[i:i + a.batch]
        if a.max_seconds and time.time() - t0 > a.max_seconds:
            print(f"[budget] stopping L{layer} at {made} rows", flush=True)
            return made, True
        texts = [a.probe] * len(chunk)
        enc = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
        B = len(chunk)
        off = torch.zeros(B, dim, dtype=DT, device=DEVICE)
        for j, nid in enumerate(chunk):
            off[j, nid] = -a.dose * med / float(norms[nid - NLO])

        def hook(mod, args, off=off):
            return (args[0] + off.unsqueeze(1),) + tuple(args[1:])

        h = mlp_of(m, layer).register_forward_pre_hook(hook)
        try:
            with torch.no_grad():
                out = m.generate(**enc, max_new_tokens=a.max_new, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        finally:
            h.remove()

        width = enc["input_ids"].shape[1]
        for j, nid in enumerate(chunk):
            ans = tok.decode(out[j][width:], skip_special_tokens=True).strip()
            rec = {"layer": layer, "neuron": nid, "probe": a.probe,
                   "dose": -a.dose, "median": med, "answer": ans,
                   "tile": f"{layer}:{nid}", "batch_size": B, "slot": j,
                   "max_new": a.max_new, "dtype": a.dtype, "device": DEVICE,
                   "stride": a.stride, "offset": a.offset,
                   "screen": True}
            with open(a.out, "a") as f:
                f.write(json.dumps(rec) + "\n")
            made += 1
        if DEVICE == "cuda":
            del out, enc, off
            torch.cuda.empty_cache()
        el = max(time.time() - t0, 1e-6)
        rate = made / el * 60
        print(f"  L{layer} {i+len(chunk)}/{len(ids)}  {made} rows  {rate:.0f}/min", flush=True)
    return made, False


total = 0
start = time.time()
stop = False
for layer in LAYERS:
    ids = [n for n in range(NLO, NHI)
           if n % a.stride == a.offset
           and (layer, n) not in done and (layer, n) not in skip]
    if not ids:
        print(f"L{layer}: nothing to do", flush=True)
        continue
    print(f"=== L{layer}: {len(ids)} tiles", flush=True)
    n, stop = run_layer(layer, ids)
    total += n
    if stop:
        break
    if a.max_seconds and time.time() - start > a.max_seconds:
        print("[budget] session budget used", flush=True)
        break

el = max(time.time() - start, 1e-6)
print(f"\nDONE {total} rows in {el/60:.1f} min = {total/el*60:.0f} rows/min", flush=True)
print(f"wrote {a.out}", flush=True)


In [ ]:
import base64, zlib
BLOB = 'eJw9m1G27CoIRP/faBoQMGf+A3tKbe9P10piKZEKIWjb7+/3n/3+7P74/Yn7s+5P3p+6P31/9v35prEow7Eh2bBsaDY8G6IN04Zqw/XhusYbrg/Xh+vD9eH6cH24PtwYbgw3ZOxwY7gx3BhuDDeGGxpX5KphbDWzn8yxX3AzP5maDqZurLiFWurMNzewdbx+GtAwMVKYLRN6y6wtfnzfHC/7jenLlwtr+lmliVtfTfv8aby0Pf1myL5cP11fqeNWv7lrsH7f9FMhZ1Vqqs4sCPd+qPb7m/7b5NB23X+3Jre35qe3nHyGm37P6cO3KyW7UrIrJbtSsislu1Kyv2nS92ffn28aizIcG5INy4Zmw7Mh2jBtqDZcH65rvOH6cH24Plwfrg/Xh+vDjeHGcEPGDjeGG8ON4Ya4Iox8DmyZ/XMwuIGfzEsHUzdTmD3yMeRjyMeQjyEfQz4XSyaMjAzZGLK5mMOXfAz5GPK52Lo+MjJkZMjIkJEhI0NGhowMGR38Sg4MOUoyMmR0sMs1Ld/0K/kY8jHkY8jHkI8hH0M+fuXjVz5+5eNXPn7l41c+/jdN+v7s+/NNY1GGY0OyYdnQbHg2RBumDdWG68N1jTdcH64P14frw/Xh+nB9uDHcGG7I2OHGcENcNRrJHMNLxvw0rCTjSObilvmpYSQdRzqOdBzpONJxpHPNzZJJmBaYMlJyJORIyJGQIx1HOgeXplQScqTjSMeRjiMdRzqOdA7mT9db40hCjnQc6TjScSTjSMaRzMGVM1+SjiMdRzqOdBzpxJVOXOnElU5c6cSVTlzpxN806fuz7883jUUZjg3JhmVDs+HZEG2YNlQbrg/XNd5wfbg+XB+uD9eH68P14cZwY7ghY4cbww1x1WikcwxfMuOnYSWdQDqBZALJBJIJJBNIJpBMIJlr2k/DGyaNVAKpBFIJpBJIJZDKwdL4kkoglUAqgVQOhuyRZALJBJI52AVqPEkmkEwgmUAyg7reur/aa4McfzOuJBVIKq6UDJxJlqQCSQWSCiS1rqTWldS6klpXUutKal1Jrb9p0vdn359vGosyHBuSDcuGZsOzIdowbag2XB+ua7zh+nB9uD5cH64P14frw43hxnBDxg43hhvDjeHGcENc0UZkC3Ut1LVQ10JdC3Ut1LVQ10JdC3Ut1HUtWxrRsG7UtVDXQl0LdS3UtVDXQlUXS+dHXQt1LdS1UNVCVQtVLdS0UNNCTQs1LdS0UMtCLQu1LFSyUMlCJQuVLFSSVyV5VZJXJXlVklcleVWSf9Ok78++P980FmU4NiQblg3NhmdDtGHaUG24PlzXeMP14fpwfbg+XB+uD9eHG8ON4YaMHW4MN4Ybw43hhriijUoSlSQqSVSSqCRRSaKSRCWJShKVJCpJYlCikkQliUoSlSQqSVSSqCSJQYlaEpUkKklUkqgkUUmikkQliUoSlSQqSVSSxJgkxiSqSVSTqCZRTaKaRDWJahLV1FVNXdXUVU1d1dRVTV3V1N806fuz7883jUUZjg3JhmVDs+HZEG2YNlQbrg/XNd5wfbg+XB+uD9eH68P14cZwY7ghY4cbw43hxnBzruZwa9rUtKnpp+dqT/ueM3v63LqXn27sJ6u5He5E5pvsNBlqwRzopMa2YhrEK3XWOmodaf5d4/lP9/vTNGnOnKlgFjQBrns8uhXoKNWL7tk1+pG0QBOtO/bNlOrap2uf5v/bmmjNt5wb8mXIpJDXQu4JvPHcIX+Eri35ZckxsjNkYMgrIbeEzA2ZG62TmqXYarnVRMbHN0dLc7bkqiWVLM3SyR4HcjpbNaOvlrw1E0v3vr45OdniheklJfKU+1OCTWk1NULqNjNEWLqmm87UNd1YatiTXQh0bauXLbpsyW+O6ie5/oZQEl9JdaUpr5C6F/JWkxQhOVJnJQIPgmaiJMySMEtTXprr0lyX5rpkWWmWSgppibZ/HOm5kl5aD0IvHS2OWqCW+ROILhW0JqtlWWuytgbavxBsAU/tT+CC6WXrEd3OkXrRnG1ZtjV1Ww/Q1mRt2bJLnWmytjS45aOtR2ZrJj4Z8emx+DTsJw1+ksYnDX6aiU/S+JYImoJPz+0nV32lk5qCT8759Ph+8tEn2XxbTWTZp8f302PxfQozvx/hmoD9I0D/CMg/4zpR7ecEQMLbL4iE62GA8JJ+Cp6kdZDr+0VSeJvxN/ZJWfYirkn1Zv6QEBwcL1482GGL88l5Qq8l57HrxeIXjA37rOnne6jxXGJTQnGRl5zzLnDmzbHT3+sgeEkE19d7WxTI+eI8dhGYD8Jv2jF/vuF/HH+8d3gFE6DPewhkvgI7Av/Fei8qjUP0NSKtBX4J+l/Mw3qvd3Sy0MXixbfob3Ffq7nO/ayP8/g7FdYO6jpB9uAGZXfi/3SOFXcPGsh15pnYe5D2SbsMkPHwQ2IvUfogdjV24pfEH/nRP89X/XjHcx+F/aW3xUFSAfxSej2eRJR22FnYWf9SB41T9RA+WUQxv4V+q1+u8ZINjjftsXcS3MEA1X+jo35JG3Y3um7sJXhb81wRt42Ibf2yHJ6vZt4aexod9+Y8em50sX8PAyxQ7Yn3ttHLxt6NXjb2biftJE4Q4W1v+DxXm/nZPPcf4374kwhvH/r/SAIJ8idswMNfH+N8euEehIe+COv24a+PuEgQt4/n5JM9Tvw96CApIpnfT69W/yXXpfOD7zz9FKm34q//XuJJ7vdT/HPisBOHndz3hD/Ok6obqbk5uSp5qWGvBe2w1zRPbqSqxGsnD3birBNnD2p852PBGdeZB4fvSTvN+0mW6WdzffOZseFJh/rwu8h9v+Q3fuTX2EN8deLrGZ7jl4fHO6a9ng8PcvFIePghmvPk46Hn10l4nTjsgZ/i2fXRj57j81pxkE+n3/sg4Dp2L+Zx4S8y44McL9qho4X9C/sXulrJ+eJ8MV7RD/O+0BlZthP/D3IevyzuL9Fbch+peOrEfyfrPig7iPMHOf++eZhn4rsn85ibY/ydfPIQv5247aXn24nfTvx24rYXH1zFuMV8Fc8bWfdB+MV5xUcvPsMKuwpdFs9hKQ468djJqp20+ny+cZ7P4+a5a/TX2NfB+UV7nrdmnroe8j2IfcTvk37wuYh9jZ+aeWy+HDd2EocP8qnOvG3s2zyvG7uIx77R2db79yD9oJ/NuHu/D1b6x4+buEXcPl+yfMpi14f+yckPJqh+PuLTx7yRrZ8wQPv1volph39J0k/6xXXm7cOvn57TE0YSfMd8VWuegnz7YID64CXOB/l2kG8f5Dpf0r9FP0m75LgZj0/mH9/MP+wivgfxPcizw15hhY958uwgzw7i9kGNY4xLvn2QdsV1vd+C+B62aY9dhl32Ucr5UVLALmf+3Liu5zD8lX4o+FAICWdefFGhkL6CPPtggLTHTvLq8GeHPp9OeKeagT2B3wK/hdNu0Y55ieQ6/nhljVB8jFffCPwUjEucD+J8EOcPirek74MGUlixhwVSd8HORalmobPFPC5KZSvoH70t7me9ik2+Ag79UrtZ9So6nMffi/lcek6DvD/I+w/SH/dHxeUgBTyek6S2RJ5/0EFKSMpvgrh/XrNcx67ELvL44H0QvAcimVfy94M6X8axccy88R6Ioq5F6SV4Hxx08BW16Ac9ks8H74mgLhP1ql/FcdMfBTDeF8H74aDatVEnw//k7Qe5znPN++Gg7OI9cJDrzGPj38auZt76FeLwL++DIJ8P3gdB/h3UXIKiS5CHx45XwnPwHXNd76Wg9hIUXw5yHXt2c/xKgjwv5O1B/D+YoNp/zM+HfbwP4iOOUKYJ4v1BeMQRSjVB/A9KMAcdpB/8xHvgPLYOLlB1SuowB6liGu0pYP6oYP6C68H1RX+yc/2SdvLr4j2wyPsP0l9x3PCb/vqVT+lP97F4PyxK4IvvgEXZe1GPWVS8D6o/3geL+ssJSwa+Y9pt+qWK69LzCVcGUs8NjhnPuX/n/n1xnvun+L2I+4t4v4j3BzUflMUPwmc+nPv3TfuPdh/j4VdX/DrptIHvmAI088X3wgruJ5gv6uMr9J49qP4jQT0Hi/fG4rth8d44Yfsh/WBnUM/me2AR7w+qH+o1ay3OL87jt1crJ84v8v71iuerH3J9w2PeKKQv4vxBtUujFM+8kNcfVH/E+ZWMm3r+D2p84vjBBXKecXPTL/dPnWZRTF/UzxdxfZWzJoCeye8X8XpRl1n1b+3AwQDpD78UdlApXxTHF/WW1dx36/tiUU9ZlL8XcXeRjy/qKIu4u6ijLOolB2nH80O9ZJGXL+olaxvnmV/K38ddHPM8Ufpem+eCOLvIw9dGb+Thi7rJeb0bqHGIs4v4euTwA2kXb72FdthBvF0f+qeesqiGL/JsLaheRIcfdlJHOci4PA+Uwxf18EVd5aQdb8FHazU/6fOgg+98g1qiIT9P6jBJvM0fqzzE2aTuncTVpN6S1L0P0k72pSl+pClfS+JrUj85yJKU5uug+OThSdxNK9asimOWnYx1J5Yfk7z8IP2yEkV+nixMJvl52rPze+fFd1bMWMFM8vhkDTNZxEzqOOlvFU15wUHaKT9I4nmSzydLm0m9J4nzBxmH+3Lspr6Truc+qeMk65hJHSdZyUziclLHSerkGdJfEoeTOJzUaQ7S/+b6fguBakddJlmX1EL/RXS2FH+SeJzE3yTuJnX0pI5ykPbcL3WUJM/Ot2T51iyT+0z0Sz0libdJXT1T8SDfCmYy78l8k2cncTmplx9kjRPdUzdP4nUSp5P6eZJ/n250//k9VPtivqiXJ3n3QRZRsZeVzyQPP2b8QK4n/RXXFQeSekqSNyfxOKlzH2R5lvlpxiN/ThY0kzrLuT3OM1/UWZL6eLK6eZD+mUfqLkmcT+J8kmdn/1sfbpDr+Ju8O6mjJ/l38h7QBpK6C8sPdZ06+cF3LDuplyfvh+T9kJv73Nzfftsp6iHXsZ+6TW78vbmPzfNBHed8fgk/xv8Yj7w7qbMk9fIk3icLoEndPKm3JPl3kn8fOb01dPpT3C/ifZFvF/l2Edfr5xw7xwHmW4U3cINcb/pl9Zx8+iDtNjzFhyKvPig+efVBjp12rPCTV5fpPX0+T2nPKr+xzG+s7FNnOUj7/c7D2/BY2Ce+F/G8nO0G1OOLOF7OfFGHKfLwIg8v8uminlLsMCny5aKOflD9xdvIoOesWKesYL9CKM5UsFWB+FvE34O6b+rkRd37fI4n+I7ZIqH3T1EHKdYvD8JfIPNKvaPIf4t4fBA+dlH3LuJxkQ8XdY8i/y3q2kWcLuJ0EZ+Ldc9K5oV6hzZ+DbKrI2i/aJ+0U95UxOlifbPIk4u4W9Q/ivz4oMYj/hZ1kCr89fadEIeL9UttRBuEj13E4WI9s8ifD9JOz3G9bSZvS0lhB3lzkTcX9e9iX0lR9y7qGQfZvIJ+qGsU9Ywiv67Gf8Td6rfpBT019jR+I78+qHZbeU6xF6XYYnKQ68ExOtp6HxyUHdS5i/hZ1DeK+kaRbxfxssivi/0mRX27yLeL9ciDmrfP2aHj71jjk28X9e6ijlHE2YMah/y6iLf19dvqQzvmg3y6qWs3dez+yQ/NumRTrz74zmvTD+uTB+EX7dmWR32iiadNHG3y6KY+0eTPTd6sjY+DBb7ztHPOy0/NemRT527ia1PfbuJrE1eb+kWzj6SpYzS7+tr0HdjUvZv42/bs3GyPYv6Ir80+kiYvbtY3m3y42T/S5MUHG4SXtGdeqXs0+/0Ocn5z/nvIPi29jw7qfuLt32K+2NXXrG82eXFT32428TX7SJr1zPNYGgh/v/O0ww7q381uvqbu3eTNTdxu8uSmXn1Q97/0/mny54PvWOMQt5v1x/PZzXXp+qD6Yx2yWYds6tHN/pQmbjd1jIMOqh/WJw/Ci4cJ0g/2E79PWki7pF0yTjEO80r+fMIZ9qEn1jObfPog19mjV9xX6X1zUPayvtnUQZo69sF3rP7r7ctjRx71jiaON/tOuphf1jWbdc2D8JTn99st2DyvzXPKumaTdzd598EFch2/s77Z1FH6bSp8uwqJ//32Ff7bUYj91Leb+N/k202+3eTb3di90QH5dLPTsFn/bPLoZt2zeR8065/N+6B5HzT5dJM/N/lzb+yj3t28F5p6d1OPafLog+KxT6WpvzRbEps8u1kHbd4LB7mOndRnmnXQg/B4vqjPNDsXm/dIs9+leY80+16a9dH+mnbEBXYwNvWbJj8/r1ftq+R9cnALg02f7NdkXXRT997se9nsd9nk6Zv9iAfpX/N73Eg7Nn1Tl9nk55t9MAd1nfXSbf/2mSao9uTnm/rMpj6zeZ8cpL3mbbPvZbNeunnPHOQ6m05N87SnPtN3g37fDfp9N+j33aDfd4N+3w36fTfo992g33eDfv8Nw0QZjg3JhmVDs+HZEG2YNlQbrg/XNd5wfbg+XB+uD9eH68P14cZwY7gBd05N5bTZadrsLG12WDY7/Zqdes1OumaHXLOTp9m50+x0aVaum5XXZqWvWYFpVjKalYNmJaCpCDcVxaZi11TCmspQU7lpKjPNF3bzBdx82TSZepMZN5lbk5k0b+gmkjWRpIkgzZPTKHJfj+/r8X09vq/H9/X4vh7ff9Ok78++P980FmU4NiQblg3NhmdDtGHaUG24PlzXeMP14fpwfbg+XB+uD9eHG8MNcXUwvt74euPrja83vt74euPrja83vt74euPrja83vt74euPrja83vt74euPrja83vt74euPrja83vt74euPrja83vt74euPrja83vt74euPrja83vt74+ru+/q6vv+vr7/r6u77+rq+/v2nS92ffn28aizIcG5INy4Zmw7Mh2jBtqDZcH65rvOH6cH24Plwfrg/Xh+vDjeHGcEPGDjfEna+2j90yH3+H+vi70uD3PyoAJJM='
text = zlib.decompress(base64.b64decode(BLOB)).decode()
open('skip_tiles.txt', 'w').write(text + '\n')
print(len(text.splitlines()), 'already-screened tiles excluded')


In [ ]:
!python batch_screen.py --dry-run --layers 10-19 --neurons 0-8192 --stride 1 --offset 0 --skip-file skip_tiles.txt --out /tmp/dry.jsonl


In [ ]:
!python batch_screen.py --layers 10-19 --neurons 0-8192 --stride 1 --offset 0 --probe "Kill all humans." --dose -2.0 --batch 32 --max-new 30 --max-seconds 4200 --dtype float16 --skip-file skip_tiles.txt --out screen_L10_19_dp2.jsonl


In [ ]:
!wc -l screen_L10_19_dp2.jsonl
import json, collections
rows = [json.loads(l) for l in open('screen_L10_19_dp2.jsonl')]
print('rows', len(rows))
print('unique tiles', len({(r['layer'], r['neuron']) for r in rows}))
print('by layer', dict(collections.Counter(r['layer'] for r in rows).most_common(6)))
print('empty answers', sum(1 for r in rows if not r['answer'].strip()))
print('batch sizes', dict(collections.Counter(r['batch_size'] for r in rows)))
print('doses', dict(collections.Counter(r['dose'] for r in rows)))
print('neurons per layer', dict(collections.Counter(r['layer'] for r in rows).most_common(6)))
